# Wrist-signal ablation with the 1D-CNN (leave-one-subject-out, WESAD)

Question: how much does each wrist signal (BVP, EDA, ACC, TEMP) contribute to stress detection, and could EDA
(or another signal) be dropped on the device?

For every configuration below a **dedicated** 1D-CNN is trained from scratch on 14 subjects and tested on the
held-out one, for all 15 subjects (same protocol as the earlier "dedicated" runs: binary stress vs non-stress,
20 epochs, no modality dropout, last epoch evaluated). Configurations:

* `all` (baseline), `no_bvp`, `no_eda`, `no_acc`, `no_temp` (drop one signal)
* `only_bvp`, `only_eda`, `only_acc`, `only_temp` (one signal alone)
* `bvp_eda`, `bvp_acc` (the two most interesting pairs)

Results are cached per fold on Drive, so if Colab disconnects, re-run the cells and it resumes.
Set `QUICK = True` first for a 2-minute smoke test.

**Files needed on Drive** (`MyDrive/licenta/`):
`stressnet_colab.zip` (the code) and `WESAD/wesad_cache/wesad_wrist.pkl` (pre-built windows, ~106 MB). If the
windows file is missing, the notebook rebuilds it from `WESAD/WESAD.zip` (a few extra minutes).

In [ ]:
import os, sys, subprocess, zipfile, time
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display
except NameError:
    display = print

QUICK = False          # True = 3 subjects, 2 epochs: only checks that everything runs
SEEDS = [0]            # add seeds (e.g. [0, 1, 2]) for more reliable numbers; runtime scales linearly
EPOCHS = 20

if IN_COLAB:
    drive.mount('/content/drive')
    BASE = Path("/content/drive/MyDrive/licenta")
    CODE_DIR = Path("/content/stressnet_code")
    with zipfile.ZipFile(BASE / "stressnet_colab.zip") as z:
        for info in z.infolist():                      # tolerate zips made on Windows (backslash paths)
            target = CODE_DIR / info.filename.replace("\\", "/")
            if info.is_dir() or info.filename.endswith(("/", "\\")):
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(z.read(info))
    assert (CODE_DIR / "src/stressnet/__init__.py").exists(), "code zip has an unexpected layout"
    sys.path.insert(0, str(CODE_DIR / "src"))
    WINDOWS_CACHE = BASE / "WESAD/wesad_cache/wesad_wrist.pkl"
    WESAD_ZIP = BASE / "WESAD/WESAD.zip"
    WESAD_ROOT = Path("/content/wesad_data/WESAD")
    RESULTS_DIR = BASE / "WESAD/wesad_cache/ablation_cnn"
else:
    ROOT = Path(os.environ.get("STRESSNET_ROOT", ".")).resolve()
    WINDOWS_CACHE = ROOT / "cache/wesad_wrist.pkl"
    WESAD_ZIP, WESAD_ROOT = None, ROOT / "data/WESAD"
    RESULTS_DIR = ROOT / "cache/ablation_cnn"
if QUICK:
    RESULTS_DIR = RESULTS_DIR.parent / "ablation_cnn_quick"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from stressnet.constants import LABEL_MAP
from stressnet.data import wesad
from stressnet.data.quality import apply_gate
from stressnet.data.splits import to_binary_windows
from stressnet.training import get_device, loso_deep_cv

DEVICE = get_device()
print("results dir:", RESULTS_DIR)

## 1. Load the windows
60 s windows, 15 s step, baseline-normalised per subject, wrist signals only (the chest-based clipping flags are
already applied). 2,733 windows from 15 subjects are expected after the clip gate.

In [ ]:
if IN_COLAB and not Path(WINDOWS_CACHE).exists():
    print("windows cache not found -> rebuilding from the raw WESAD zip (a few minutes)")
    subprocess.run(["unzip", "-q", "-n", str(WESAD_ZIP), "-d", "/content/wesad_data"], check=True)

windows, report = wesad.load_or_build_final_dataset(
    WINDOWS_CACHE, WESAD_ROOT, LABEL_MAP, keep_chest=False)
final_windows, _ = apply_gate(windows, clip=True)
wins = to_binary_windows(final_windows)

if QUICK:
    keep = sorted({w["subject"] for w in wins})[:3]
    wins = [w for w in wins if w["subject"] in keep]
    EPOCHS = 2

subjects = sorted({w["subject"] for w in wins}, key=lambda s: int(s[1:]))
print(len(wins), "windows |", len(subjects), "subjects:", subjects)
print("stress windows:", sum(w["model_label"] for w in wins), "| non-stress:", sum(1 - w["model_label"] for w in wins))

## 2. Configurations
Ordered so the most important comparisons (`all` vs `no_eda`) finish first.

In [ ]:
SIGNALS = {"bvp": "wrist_bvp", "eda": "wrist_eda", "acc": "wrist_acc", "temp": "wrist_temp"}
ALL = list(SIGNALS)

def mods(*names):
    return [SIGNALS[n] for n in names]

CONFIGS = {"all": mods(*ALL)}
for s in ["eda", "temp", "acc", "bvp"]:
    CONFIGS[f"no_{s}"] = mods(*[x for x in ALL if x != s])
for s in ["bvp", "eda", "acc", "temp"]:
    CONFIGS[f"only_{s}"] = mods(s)
CONFIGS["bvp_eda"] = mods("bvp", "eda")
CONFIGS["bvp_acc"] = mods("bvp", "acc")

for k, v in CONFIGS.items():
    print(f"{k:10s} {v}")
print(f"\n{len(CONFIGS)} configs x {len(subjects)} subjects x {len(SEEDS)} seed(s) = "
      f"{len(CONFIGS) * len(subjects) * len(SEEDS)} model trainings, {EPOCHS} epochs each")

## 3. Run the leave-one-subject-out ablation
Each fold's result is cached in `RESULTS_DIR`; finished folds are skipped on re-run.

In [ ]:
results = {}
t0 = time.time()
for cfg, mod_list in CONFIGS.items():
    for seed in SEEDS:
        print(f"=== {cfg} (seed {seed}) | elapsed {(time.time() - t0) / 60:.1f} min")
        results[(cfg, seed)] = loso_deep_cv(
            mod_list, "cnn", f"abl_{cfg}_s{seed}", wins, num_classes=2, epochs=EPOCHS,
            batch_size=32, lr=1e-3, weight_decay=1e-2, dropout=0.3,
            cache_dir=RESULTS_DIR, select_best=False, device=DEVICE, seed=seed)
print(f"done in {(time.time() - t0) / 60:.1f} min")

## 4. Results
Per subject, scores are first averaged over seeds; then mean/std are taken over subjects.
`vs all` is the paired per-subject difference to the four-signal model with a Wilcoxon signed-rank p-value
(small sample: 15 subjects, treat p-values as a rough guide).

In [ ]:
rows = []
for (cfg, seed), folds in results.items():
    for f in folds:
        m = f["subsets"]["full"]
        rows.append({"config": cfg, "seed": seed, "subject": f["held_out_subject"],
                     "macro_f1": m["macro_f1"], "balanced_accuracy": m["balanced_accuracy"]})
raw = pd.DataFrame(rows)
per_subj = raw.groupby(["config", "subject"])[["macro_f1", "balanced_accuracy"]].mean().reset_index()
piv = per_subj.pivot(index="subject", columns="config", values="macro_f1")
piv = piv.loc[subjects, [c for c in CONFIGS if c in piv.columns]]

summary = pd.DataFrame({
    "macro_f1": piv.mean(), "std": piv.std(), "worst_subject": piv.min(),
    "bal_acc": per_subj.groupby("config")["balanced_accuracy"].mean().reindex(piv.columns),
})
def paired(cfg):
    if cfg == "all" or "all" not in piv:
        return pd.Series({"vs_all": np.nan, "p": np.nan, "better_in": np.nan, "worse_in": np.nan})
    d = piv[cfg] - piv["all"]
    try:
        p = wilcoxon(piv[cfg], piv["all"]).pvalue
    except ValueError:
        p = np.nan
    return pd.Series({"vs_all": d.mean(), "p": p, "better_in": int((d > 1e-9).sum()), "worse_in": int((d < -1e-9).sum())})
summary = summary.join(pd.DataFrame({c: paired(c) for c in piv.columns}).T)
display(summary.round(3))
summary.round(4).to_csv(RESULTS_DIR / "ablation_summary.csv")
raw.to_csv(RESULTS_DIR / "ablation_per_fold.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(summary))
ax.bar(x, summary["macro_f1"], yerr=summary["std"], capsize=3, color="#4c72b0")
if "all" in summary.index:
    ax.axhline(summary.loc["all", "macro_f1"], color="#c44e52", ls="--", lw=1, label="all four signals")
ax.set_xticks(x); ax.set_xticklabels(summary.index, rotation=30, ha="right")
ax.set_ylabel("macro-F1 (mean +- std over subjects)"); ax.set_ylim(0, 1.05); ax.legend()
ax.set_title("1D-CNN, LOSO on WESAD: which wrist signals matter?")
fig.tight_layout(); fig.savefig(RESULTS_DIR / "ablation_bar.png", dpi=130); plt.show()

fig, ax = plt.subplots(figsize=(10, 5.5))
im = ax.imshow(piv.values, aspect="auto", cmap="viridis", vmin=0.3, vmax=1.0)
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=30, ha="right")
ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
ax.set_title("macro-F1 per held-out subject"); fig.colorbar(im, ax=ax)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "ablation_per_subject.png", dpi=130); plt.show()

## 5. Read-out
How to interpret (fill in after running):

* **Importance of a signal** = how much `no_<signal>` falls below `all`, and how well `only_<signal>` does alone.
* **Is EDA worth its hardware cost?** compare `all` vs `no_eda`, and `bvp_eda` vs `bvp_acc`.
* **Noise level:** if the std over subjects (or the p-values) is large compared with the differences, the
  ablation cannot separate the configs; add seeds (`SEEDS = [0, 1, 2]`) before drawing conclusions.